# ADK: Multi-Agent Collaboration via A2A Protocol (March 2026 Suite)

[![Open In Colab](https://colab.research.google.com/github/maruti123/partner-demos/blob/main/partner-demos-march-2026/adk_multi_agent_a2a_demo.ipynb)](https://colab.research.google.com/github/maruti123/partner-demos/blob/main/partner-demos-march-2026/adk_multi_agent_a2a_demo.ipynb)

This notebook shows how two ADK agents collaborate over the **A2A (Agent-to-Agent) Protocol** — one acting as a manager, the other as a remote specialist.

## What You'll See
1.  **Serve**: Turn any ADK agent into an A2A endpoint with `to_a2a()` — including **lifespan hooks** for startup/shutdown.
2.  **Connect**: The manager discovers the specialist via its Agent Card URL.
3.  **Observe**: A **request interceptor** logs every cross-agent message for observability.
4.  **Collaborate**: The manager delegates a task and summarizes the specialist's findings.

### Use Case
A legal review requires a **Manager Agent** (orchestrator) and a **Legal Specialist** (domain expert). Using A2A, the manager discovers the specialist, delegates a contract risk assessment, and summarizes the findings — all over the open A2A protocol.

### Release Notes
- [ADK v1.28.0](https://github.com/google/adk-python/releases/tag/v1.28.0) — `lifespan` parameter for `to_a2a()`, new A2A-ADK integration extension
- [ADK v1.27.0](https://github.com/google/adk-python/releases/tag/v1.27.0) — New `RemoteA2aAgent` implementation, A2A request interceptors

### Requirements
- `google-adk >= 1.28.0` and `a2a-sdk >= 0.3.25` installed.
- Gemini 3.1 Pro (Preview) access.

In [ ]:
# 1. Setup and Authentication
%pip install "google-adk>=1.28.0" "a2a-sdk>=0.3.25" google-genai nest-asyncio uvicorn starlette --quiet --index-url https://pypi.org/simple

try:
    from google.colab import auth
    auth.authenticate_user()
    print('Authenticated via Colab')
except ModuleNotFoundError:
    print('Not running in Colab — using Application Default Credentials (ADC)')

import os
import nest_asyncio
nest_asyncio.apply()

project_id = 'YOUR_PROJECT_ID'  # @param {type:"string"}
location = 'global'  # @param {type:"string"} — Gemini 3.1 Pro Preview requires 'global'
os.environ["GOOGLE_CLOUD_PROJECT"] = project_id
os.environ["GOOGLE_CLOUD_LOCATION"] = location
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"

### 2. [PREREQUISITES] Define the Specialist Agent

The specialist has a tool (`get_risk_rubric`) that returns a risk assessment rubric loaded during server startup via the **lifespan hook**. This makes the lifespan load-bearing — the rubric is only available after the server's startup phase completes.

In [ ]:
from google.adk import Agent, Runner
from google.adk.sessions.in_memory_session_service import InMemorySessionService
from google.genai import types

# Risk rubric — populated by the lifespan hook at server startup
risk_rubric = {}

def get_risk_rubric() -> dict:
    """Return the risk assessment rubric loaded at server startup.
    Use this before analyzing any contract to get the scoring criteria.
    """
    if not risk_rubric:
        return {"error": "Rubric not loaded — server lifespan hook has not run."}
    return risk_rubric

# The Specialist Agent (Domain Expert) — uses the rubric tool
legal_specialist = Agent(
    model="gemini-3.1-pro-preview",
    name="LegalSpecialist",
    instruction="""You are a legal expert specializing in commercial contracts.
    ALWAYS call get_risk_rubric first to load the scoring criteria, then use those
    criteria to provide a structured risk analysis for any document provided.""",
    tools=[get_risk_rubric]
)

print(f"Specialist agent '{legal_specialist.name}' defined with rubric tool.")

### 3. Core Feature: Serve the Specialist via A2A Protocol

`to_a2a()` turns any ADK agent into a Starlette HTTP app that speaks A2A. 

**New in v1.28.0**: The `lifespan` parameter lets you run startup/shutdown logic — initialize DB connections, load resources, warm caches — before the agent starts serving requests.

In [ ]:
import threading
import uvicorn
import time
import asyncio
import socket
from contextlib import asynccontextmanager
from starlette.applications import Starlette
from google.adk.a2a.utils.agent_to_a2a import to_a2a

def is_port_in_use(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('localhost', port)) == 0

A2A_PORT = 8765
if is_port_in_use(A2A_PORT):
    A2A_PORT = 8766

# Lifespan hook (v1.28.0) — loads the risk rubric at startup
# In production: init DB pools, load prompt registries, warm model caches
@asynccontextmanager
async def specialist_lifespan(app: Starlette):
    # Startup: load resources the agent needs
    risk_rubric.update({
        "categories": ["liability", "indemnification", "termination", "IP ownership"],
        "severity_scale": {"low": 1, "medium": 2, "high": 3, "critical": 4},
        "threshold": "Flag any clause scoring >= 3",
        "version": "2026-Q1"
    })
    print(f"[LIFESPAN] Rubric v{risk_rubric['version']} loaded ({len(risk_rubric['categories'])} categories)")
    yield
    # Shutdown: release resources
    risk_rubric.clear()
    print("[LIFESPAN] Rubric cleared — specialist shutting down")

specialist_app = to_a2a(
    agent=legal_specialist,
    host="0.0.0.0",
    port=A2A_PORT,
    lifespan=specialist_lifespan  # v1.28.0: startup/shutdown hooks
)

def run_a2a_server():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    config = uvicorn.Config(
        app=specialist_app,
        host="0.0.0.0",
        port=A2A_PORT,
        log_level="warning"
    )
    server = uvicorn.Server(config)
    loop.run_until_complete(server.serve())

server_thread = threading.Thread(target=run_a2a_server, daemon=True)
server_thread.start()
time.sleep(3)

# Verify lifespan ran by checking the rubric
if risk_rubric:
    print(f"✓ Lifespan confirmed: rubric v{risk_rubric['version']} loaded with {len(risk_rubric['categories'])} categories")
else:
    print("✗ WARNING: Lifespan did not run — rubric is empty")

print(f"A2A Server running at http://localhost:{A2A_PORT}")
print(f"Agent Card: http://localhost:{A2A_PORT}/.well-known/agent-card.json")

### 4. Core Feature: Manager Connects via RemoteA2aAgent

The Manager uses `RemoteA2aAgent` as a sub-agent. It only needs the agent card URL — A2A handles capability discovery, the handshake, and multi-turn delegation.

**New in v1.27.0+**: A **request interceptor** adds observability to every A2A call. `RequestInterceptor` is a Pydantic model with callable fields (not a class to subclass) — you pass standalone async functions matching the exact signatures from ADK source. The `use_legacy=False` flag enables the new A2A-ADK integration extension.

In [ ]:
from typing import Union
from google.adk.agents.remote_a2a_agent import RemoteA2aAgent
from google.adk.agents.invocation_context import InvocationContext
from google.adk.events.event import Event
from google.adk.a2a.agent.config import (
    A2aRemoteAgentConfig, RequestInterceptor, ParametersConfig
)
from a2a.types import Message as A2AMessage
from a2a.server.events import Event as A2AEvent

# --- Audit interceptor (v1.27.0 feature) ---
# RequestInterceptor is a Pydantic model with callable fields — not a class to subclass.
# We define standalone async functions matching the exact signatures from ADK source.

audit_log = []

async def audit_before_request(
    ctx: InvocationContext,
    a2a_request: A2AMessage,
    params: ParametersConfig,
) -> tuple[Union[A2AMessage, Event], ParametersConfig]:
    """Assign a request ID and start latency timer before each A2A call."""
    request_id = ctx.invocation_id[:8]  # Use ADK's own invocation ID for tracing
    ctx.session.state["_audit_start_time"] = time.time()
    print(f"  [AUDIT] >> Request {request_id} sent to specialist")
    return a2a_request, params  # Pass through — return an Event here to abort

async def audit_after_request(
    ctx: InvocationContext,
    a2a_response: A2AEvent,
    event: Event,
) -> Union[Event, None]:
    """Measure latency and record an audit entry after each A2A call."""
    request_id = ctx.invocation_id[:8]
    start_time = ctx.session.state.get("_audit_start_time", time.time())
    latency_ms = round((time.time() - start_time) * 1000, 1)
    entry = {"request_id": request_id, "latency_ms": latency_ms, "status": "success"}
    audit_log.append(entry)
    print(f"  [AUDIT] << Request {request_id} completed in {latency_ms}ms")
    return event  # Pass through — return None here to suppress the event

audit_interceptor = RequestInterceptor(
    before_request=audit_before_request,
    after_request=audit_after_request,
)

# Connect to the specialist via A2A protocol with interceptor
remote_specialist = RemoteA2aAgent(
    name="LegalSpecialist",
    agent_card=f"http://localhost:{A2A_PORT}/.well-known/agent-card.json",
    description="Remote legal specialist agent for contract review and risk analysis.",
    config=A2aRemoteAgentConfig(
        request_interceptors=[audit_interceptor]  # v1.27.0: audit hooks
    ),
    use_legacy=False  # v1.28.0: new A2A-ADK integration extension
)

# Define the Manager (Orchestrator) with the remote specialist as a sub-agent
manager_agent = Agent(
    model="gemini-3.1-pro-preview",
    name="ProjectManager",
    instruction="""You are a project manager. When asked to review contracts or assess legal risks,
    delegate the task to the LegalSpecialist agent and summarize their findings for the user.""",
    sub_agents=[remote_specialist]
)

runner = Runner(
    agent=manager_agent,
    session_service=InMemorySessionService(),
    app_name="a2a_demo",
    auto_create_session=True
)

async def run_a2a_workflow():
    print("--- Starting A2A Multi-Agent Collaboration ---")
    prompt = "Assuming a standard commercial SaaS contract, analyze common liability clause risks and provide a risk assessment using the rubric."
    print(f"User: {prompt}\n")

    message = types.Content(parts=[types.Part(text=prompt)], role='user')
    async for event in runner.run_async(
        user_id="partner_user",
        session_id="march_session",
        new_message=message
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    print(f"{event.author}: {part.text}")
                if part.function_call:
                    print(f"[SYSTEM]: {event.author} delegating to '{part.function_call.name}'")

    # Print the audit trail
    print("\n--- Audit Trail (from interceptor) ---")
    for entry in audit_log:
        print(f"  {entry['request_id']} | {entry['latency_ms']}ms | {entry['status']}")

    # Clean up httpx client (v1.28.0: proper resource management)
    await remote_specialist.cleanup()
    print("\n--- Cleanup: A2A client resources released ---")

await run_a2a_workflow()

### 5. Things to remember or know
- **Server/client model**: The specialist runs as an HTTP service (via `to_a2a()`), the manager connects via `RemoteA2aAgent`. This enables cross-team, cross-cloud, and cross-framework agent collaboration.
- **Agent card discovery**: `RemoteA2aAgent` only needs the card URL (`/.well-known/agent-card.json`). Capability exchange and security are handled by the protocol. Note: the older `/.well-known/agent.json` endpoint is deprecated.
- **Works like local sub-agents**: `RemoteA2aAgent` plugs into ADK's `sub_agents` system, so routing and delegation work the same for local and remote agents.
- **Vertex AI auth**: Set `GOOGLE_GENAI_USE_VERTEXAI=TRUE`, `GOOGLE_CLOUD_PROJECT`, and `GOOGLE_CLOUD_LOCATION` **before** importing ADK. Without these, requests go to the AI Studio endpoint and fail with `API_KEY_INVALID`.
- **Lifespan management**: `to_a2a()` accepts a `lifespan` parameter for startup/shutdown hooks. In this demo, the lifespan loads the risk rubric — without it, the specialist has no scoring criteria.
- **Request interceptors**: `RequestInterceptor` is a **Pydantic model with callable fields** — do NOT subclass it. Pass standalone async functions with exact signatures: `before_request(InvocationContext, A2AMessage, ParametersConfig) → tuple[Union[A2AMessage, Event], ParametersConfig]` and `after_request(InvocationContext, A2AEvent, Event) → Union[Event, None]`. Return an `Event` from `before_request` to abort; return `None` from `after_request` to suppress.
- **`cleanup()`**: Always call `await remote_agent.cleanup()` to close the underlying httpx client when done.
- **Runner pattern**: All March 2026 demos use the `Runner` for automatic session management and event streaming.
- **Releases**: RemoteA2aAgent in [v1.27.0](https://github.com/google/adk-python/releases/tag/v1.27.0). A2A lifespan, `use_legacy`, and extension in [v1.28.0](https://github.com/google/adk-python/releases/tag/v1.28.0).

#### `use_legacy=False`: The A2A-ADK Integration Extension

Setting `use_legacy=False` on `RemoteA2aAgent` activates the new A2A-ADK integration extension (v1.28.0). This is an internal improvement — the demo output looks the same — but it changes how A2A artifacts are converted into ADK events under the hood:

| Aspect | `use_legacy=True` (default) | `use_legacy=False` (new) |
|--------|---------------------------|--------------------------|
| **Artifact mapping** | A2A `TextPart` → raw string in ADK event | A2A `TextPart` → proper `types.Part` with full metadata |
| **Multi-modal support** | Text-only passthrough | Supports `FilePart`, `DataPart` conversion to ADK types |
| **Event fidelity** | Minimal event metadata preserved | Full A2A task status, artifact metadata carried through |
| **Type safety** | Manual extraction of response text | Native ADK types — works with `event.content.parts` directly |
| **Best for** | Backwards compatibility with pre-v1.27.0 code | New integrations, multi-modal agents, production A2A pipelines |

> **Recommendation**: Use `use_legacy=False` for all new A2A integrations. The default remains `True` only for backwards compatibility with existing deployments.